# OCR and Vision -- Reading Images with AI

This file teaches you how to extract text from images using a vision LLM.
By the end, you will know how to send images to GPT-4o-mini and get back structured text, table data, and summaries. 

## Setup

We need:
- **OpenAI client:** sends images to GPT-4o-mini.
- **Pillow (`PIL`):** creates and handles images.
- **base64:** converts image files into text strings that can be sent in API calls.
- **PyMuPDF (`fitz`):** converts PDF pages into images. 

In [3]:
!pip install -q openai pillow pymupdf python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
from dotenv import load_dotenv
import os 

load_dotenv()
API_KEY = os.getenv("API_KEY")

In [6]:
import base64
from openai import OpenAI
from PIL import Image, ImageDraw, ImageFont
import fitz

In [7]:
client = OpenAI(api_key=API_KEY)

In [8]:
MODEL = "gpt-4o-mini"

**What happened:** We imported all libraries and created the OpenAI client.

## What is OCR ?

**OCR (Optical Character Recognition)** is software that reads text from images.

Common OCR tools:
- **Tesseract:** free, open source, good for clean printed text.
- **Google Cloud Vision API:** paid, high accuracy.
- **AWS Textract:** paid, good for forms and tables. 

A **Vision LLM (like GPT-4o-mini)** can also read images. It is often better than OCR because it understands layout, handwriting, charts, and tables -- not just individual letters. 

## Send an Image to GPT-4o-mini

**What it does:** Sends an image to GPT-4o-mini and asks it to read the contents. 

**When to use it:** When you have scanned documents, photos of text, or any image that contains information you need as text. 

**Steps:**

1. Create a test image (a fake invoice).
2. Convert the image to a base64 string. 
3. Send it to GPT-40-mini with a prompt. 
4. Read the structures response. 

First, let us create a test invoice image. 

In [9]:
img = Image.new("RGB", (600, 300), "white")
draw = ImageDraw.Draw(img)
font = ImageFont.load_default()
draw.text((20, 20), "INVOICE #12345", fill="black", font=font)
draw.text((20, 50), "Date: 2024-11-15", fill="black", font=font)
draw.text((20, 70), "Customer: Acme Corp", fill="black", font=font)

In [10]:
draw.text((20, 110), "Item              Qty   Price", fill="black", font=font)
draw.line([(20, 128), (350, 128)], fill="black")
draw.text((20, 135), "Cloud Credits      10   $500", fill="black", font=font)
draw.text((20, 155), "Support Plan        1   $200", fill="black", font=font)
draw.line([(20, 175), (350, 175)], fill="black")
draw.text((20, 180), "Total:                  $700", fill="black", font=font)

In [11]:
img_path = "./tmp/test_invoice.png"
img.save(img_path)


**What happened:** We drew a simple invoice image with items, quantities, and prices.

### Convert Image to Base64

**Base64** is a way to represent binary data (like a image file) as a text string. The OpenAI API accepts images as base64 strings. 

In [13]:
with open(img_path, "rb") as f:
    raw_bytes = f.read()

encoded_bytes = base64.b64encode(raw_bytes)

image_base64 = encoded_bytes.decode("utf-8")


print(encoded_bytes[:30])
print(image_base64[:30])

b'iVBORw0KGgoAAAANSUhEUgAAAlgAAA'
iVBORw0KGgoAAAANSUhEUgAAAlgAAA


**What happened:** We read the image file as bytes and converted it to a base64 text string.

### Send to GPT-4o-mini Vision

In [15]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=500,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Extract all text and data from this invoice image. Return as structured JSON."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_base64}"
                    }
                }
            ]
        }
    ]
)

In [16]:
answer = response.choices[0].message.content

print(answer)

```json
{
  "invoice": {
    "invoice_number": "12345",
    "date": "2024-11-15",
    "customer": "Acme Corp",
    "items": [
      {
        "item": "Cloud Credits",
        "quantity": 10,
        "price": 500
      },
      {
        "item": "Support Plan",
        "quantity": 1,
        "price": 200
      }
    ],
    "total": 700
  }
}
```


**What happened:** We sent the image to GPT-4o-mini with a text instruction. The model read the image and returned the data as structured text. 

## Extract a Table from an Image

**What it does:** Sends an image to GPT-4o-mini and asks it to return table data as JSON. 

**When to use it:** When you have screenshots or scans that contain tables. 

In [17]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    temperature=0,
    response_format={
        "type": "json_object"
    },
    messages=[
        {
            "role": "system",
            "content": "Extract tables from images as JSON."
        }, 
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Return JSON: {\"headers\": [...], \"rows\":[[...], ...]}"
                }, 
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_base64}"
                    }
                }
            ]
        }
    ]
)

In [19]:
table_data = response.choices[0].message.content
print(table_data)

{
  "headers": ["Item", "Qty", "Price"],
  "rows": [
    ["Cloud Credits", 10, "$500"],
    ["Support Plan", 1, "$200"],
    ["Total", "", "$700"]
  ]
}


**What happened:** By using response_format={"type": "json_object"}, we told the model to return valid JSON. The table from the image is now structured data. 

## Convert a PDF page to an Image

**What it does:** Renders a PDF page as an image, then sends it to GPT-4o-mini.

**When to use it:** When a PDF has complex layouts, charts, or scanned pages that regular text extraction cannot handle. 

**Steps:**

1. Open the PDF with PyMuPDF.
2. Render a page as a PNG image.
3. Convert to base64. 
4. Send to GPT-4o-mini.

In [20]:
# Create a sample PDF
pdf_path = "./tmp/vision_test.pdf"
doc = fitz.open()
page = doc.new_page()
page.insert_text((72, 72), "Q4 Revenue: $12.5M (+15%)", fontsize=14)
page.insert_text((72, 100), "Customers: 2500 (+20%)", fontsize=12)
page.insert_text((72, 125), "Churn: 3.2% (down from 4.1%)", fontsize=12)
doc.save(pdf_path)
doc.close()

In [21]:
def pdf_page_to_base64(path, page_num=0, dpi=150):
    
    doc = fitz.open(path)
    page = doc[page_num]

    scale = dpi / 72

    mat = fitz.Matrix(scale, scale)

    pix = page.get_pixmap(matrix=mat)

    img_bytes = pix.tobytes("png")

    doc.close()


    encoded = base64.b64encode(img_bytes)

    return encoded.decode("utf-8")

In [24]:
page_b64 = pdf_page_to_base64(pdf_path)

print(page_b64[:50])

iVBORw0KGgoAAAANSUhEUgAABNgAAAbbCAIAAABT346sAAAACX


In [25]:
response = client.chat.completions.create(
    model = MODEL,
    max_tokens=400,
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Read this document and extract all key metrics."
                }, 
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{page_b64}"
                    }
                }
            ]
        }
    ]
)

In [26]:
answer = response.choices[0].message.content
print(answer)

Here are the key metrics extracted from the document:

- **Q4 Revenue**: $12.5M (+15%)
- **Customers**: 2500 (+20%)
- **Churn Rate**: 3.2% (down from 4.1%)


**What happened:** We converted the PDF page to an image and sent it to GPT-4o-mini. The model read the page and extracted the metrics.

## When to Use What

| Document Type | Best Approach |
|---|---|
| Clean text PDF (digital) | PyMuPDF text extraction (free, fast) |
| Scanned PDF (image-based) | GPT-4o-mini vision |
| Complex tables in PDF | GPT-4o-mini vision with JSON output |
| Charts and graphs | GPT-4o-mini vision (OCR cannot read charts) |
| Handwritten documents | GPT-4o-mini vision |
| High-volume, low-cost | Tesseract OCR (free, local) |

In [ ]:
os.remove(img_path)
os.remove(pdf_path)

## Summary

- **OCR** reads text from images. Tesseract is a free OCR tool.
- **Vision LLMs** (like GPT-4o-mini) can also read images, and they understand layout, tables, and charts better than OCR.
- To send an image to GPT-4o-mini, convert it to **base64** and include it in the message content.
- Use `response_format = {"type": "json_object"}` to get structure JSON output. 
- PDF pages can be converted to images with PyMuPDF and then sent to GPT-4o-mini. 
- For clean digital PDFs, text extraction is cheaper. For scanned or complex PDFs, vision is more accurate.